In [ ]:
!pip install 'git+https://github.com/facebookresearch/detectron2.git'
# !pip install opencv-python scikit-image pycocotools

In [ ]:
import os
import cv2
import torch
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2 import model_zoo
from google.colab import drive


from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
from google.colab.patches import cv2_imshow

import numpy as np
import json



In [ ]:
BASE = "/content/drive/Shareddrives/Comp 413 Team 3 /Notebooks/Data"

FULL_PARTIAL_MIX= "FullAndPartialBodyMixDataset-20240221T171207Z-001"
FULL = "FullBodyDataset-20240221T171203Z-001"
PARTIAL = "PartialBodyDataset-20240221T171158Z-001"

DATASETS = [
    FULL_PARTIAL_MIX,
    FULL,
    PARTIAL,
]

for d in DATASETS:
    src = f"{BASE}/{d}.zip"   # Drive location
    dst = f"/content/{d}.zip" # Local zip file
    name = f"/content/{d}" # Extration folder

    if not os.path.exists(dst):
        print(f"Copy {d}...")
        !cp "$src" "$dst"

    if not os.path.exists(name):
        print(f"Unzip {d}...")
        !unzip -q "$dst" -d "$name"

print(f"Folders available:")
!ls /content/

In [ ]:

# drive.mount('/content/drive')

cfg = get_cfg()
cfg.merge_from_file("/content/drive/Shareddrives/Comp 413 Team 3 /Notebooks/lesion_output_1/config.yaml")

cfg.MODEL.WEIGHTS = "/content/drive/Shareddrives/Comp 413 Team 3 /Notebooks/lesion_output_1/model_final.pth"

cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

predictor = DefaultPredictor(cfg)

print("Predictor is ready!")

In [ ]:

# /content/FullAndPartialBodyMixDataset-20240221T171207Z-001
img_path = "/content/FullAndPartialBodyMixDataset-20240221T171207Z-001/FullAndPartialBodyMixDataset/Data/model_001_WFemale.png"

# img_path = "/content/FullAndPartialBodyMixDataset-20240221T171207Z-001/FullAndPartialBodyMixDataset/Data/model_000_WFemale.png"
im = cv2.imread(img_path)

outputs = predictor(im)

v = Visualizer(im[:, :, ::-1], metadata=None, scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
cv2_imshow(out.get_image()[:, :, ::-1])

In [ ]:

def track_lesions_iou(masks_t1, masks_t2, iou_threshold=0.2):
    """
    Matches lesions between T1 and T2 based on spatial overlap.
    Calculates growth: ((Area_T2 - Area_T1) / Area_T1) * 100
    """
    matches = []
    matched_t2_indices = set()

    # Loop through every lesion found in Timepoint 1
    for i, m1 in enumerate(masks_t1):
        best_iou = 0
        best_match_idx = -1

        # Compare against every lesion found in Timepoint 2
        for j, m2 in enumerate(masks_t2):
            intersection = np.logical_and(m1, m2).sum()
            union = np.logical_or(m1, m2).sum()
            iou = intersection / union if union > 0 else 0

            if iou > best_iou:
                best_iou = iou
                best_match_idx = j

        # If the best overlap is decent, they are the same lesion
        if best_iou > iou_threshold:
            area1 = np.sum(m1)
            area2 = np.sum(masks_t2[best_match_idx])

            # Growth Formula: ((A2 - A1) / A1) * 100
            growth = ((area2 - area1) / area1) * 100

            matches.append({
                "t1_idx": i,
                "t2_idx": best_match_idx,
                "iou": best_iou,
                "growth_pct": growth,
                "status": "Persistent"
            })
            matched_t2_indices.add(best_match_idx)
        else:
            # No match found in T2
            matches.append({
                "t1_idx": i,
                "status": "Disappeared",
                "growth_pct": -100
            })

    # Identify any lesions in T2 that didn't exist in T1
    for j in range(len(masks_t2)):
        if j not in matched_t2_indices:
            matches.append({
                "t2_idx": j,
                "status": "New Lesion",
                "growth_pct": 0
            })

    return matches

In [ ]:

def get_gt_id(mask, metadata_path):
    """
    Finds the real LesionName from the JSON that matches the detected mask area.
    """
    if not os.path.exists(metadata_path):
        return "MetaNotFound"

    with open(metadata_path, 'r') as f:
        meta = json.load(f)

    # Find the center of the AI's detected mask
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return "NoPixels"

    cx, cy = int(np.mean(xs)), int(np.mean(ys))

    # Check which Ground Truth lesion is closest to that center
    for lesion in meta.get('Lesions', []):
        try:
            # Handles the scientific notation (e.g., -2.147E+09) and decimals
            lx, ly = [int(float(c)) for c in lesion['Coordinate'].split(',')]

            # Calculate Euclidean distance
            distance = np.sqrt((cx - lx)**2 + (cy - ly)**2)

            # If AI center is within 25 pixels of GT center, it's a match
            if distance < 25:
                return lesion['LesionName']
        except:
            continue

    return "Unknown"

In [ ]:
import matplotlib.pyplot as plt

import random
def visualize_longitudinal_results(base_name):
    # 1. Load Data/content/FullBodyDataset-20240221T171203Z-001
    img1_path = f"/content/FullAndPartialBodyMixDataset-20240221T171207Z-001/FullAndPartialBodyMixDataset/Data/model_001_WFemale.png"
    img2_path = f"/content/drive/Shareddrives/Comp 413 Team 3 /Notebooks/Data/T2_FullAndPartialBodyMixDataset/data/model_001_WFemale.png"

    img1 = cv2.imread(img1_path)
    img2 = cv2.imread(img2_path)
    if img1 is None or img2 is None: return

    # 2. Get AI Predictions
    out1 = predictor(img1)
    out2 = predictor(img2)
    masks_t1 = out1["instances"].pred_masks.to("cpu").numpy()
    masks_t2 = out2["instances"].pred_masks.to("cpu").numpy()

    matches = track_lesions_iou(masks_t1, masks_t2)

    canvas1 = img1.copy()
    canvas2 = img2.copy()

    colors = [tuple(random.randint(50, 255) for _ in range(3)) for _ in range(len(matches))]

    for i, m in enumerate(matches):
        color = colors[i]

        if 't1_idx' in m:
            mask1 = masks_t1[m['t1_idx']]
            contours, _ = cv2.findContours(mask1.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(canvas1, contours, -1, color, 3)

            M = cv2.moments(mask1.astype(np.uint8))
            if M["m00"] != 0:
                cx, cy = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
                label = f"ID:{i}"
                if m['status'] == "Disappeared": label += " [GONE]"

                cv2.putText(canvas1, label, (cx, cy-20), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

        if 't2_idx' in m:
                    mask2 = masks_t2[m['t2_idx']]
                    contours, _ = cv2.findContours(mask2.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(canvas2, contours, -1, color, 3)

                    M = cv2.moments(mask2.astype(np.uint8))
                    if M["m00"] != 0:
                        cx, cy = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])

                        if m['status'] == "Persistent":
                            if round(m['growth_pct']) != 0:
                                label = f"ID:{i} (+{m['growth_pct']:.0f}%)"
                            else:
                                label = f"ID:{i}"
                        else:
                            label = "NEW"
                        cv2.putText(canvas2, label, (cx, cy-20), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

    fig, axes = plt.subplots(1, 2, figsize=(12, 12))
    axes[0].imshow(cv2.cvtColor(canvas1, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Timepoint 1: {base_name}", fontsize=15)
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(canvas2, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Timepoint 2: {base_name}", fontsize=15)
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

visualize_longitudinal_results("001_WFemale")

###Stats for lesion detection

In [ ]:
###Stats for lesion detection
def run_full_analysis(base_name):
    """
    Ties everything together: Predicts, Matches, Verifies, and Prints a Table.
    """
    # 1. Setup Paths
    T1_IMG = f"/content/FullBodyDataset-20240221T171203Z-001/FullBodyDataset/Data/model_001_WMale.png"
    T2_IMG = f"/content/drive/Shareddrives/Comp 413 Team 3 /Notebooks/Data/T2_FullBodyDataset/data/model_001_WMale.png"
    T1_META = f"/content/FullBodyDataset-20240221T171203Z-001/FullBodyDataset/Metadata/001_WMale.json"
    T2_META = f"/content/drive/Shareddrives/Comp 413 Team 3 /Notebooks/Data/T2_FullBodyDataset/metadata/001_WMale.json"

    # 2. Run Inference
    img1 = cv2.imread(T1_IMG)
    img2 = cv2.imread(T2_IMG)

    if img1 is None or img2 is None:
        print(img1, img2)
        print(f"Error: Could not find images for {base_name}")
        return

    out1 = predictor(img1)
    out2 = predictor(img2)

    masks_t1 = out1["instances"].pred_masks.to("cpu").numpy()
    masks_t2 = out2["instances"].pred_masks.to("cpu").numpy()

    # 3. Match Lesions
    matches = track_lesions_iou(masks_t1, masks_t2)

    # 4. PRINT FORMATTED TABLE
    print(f"\n" + "="*85)
    print(f" LONGITUDINAL ANALYSIS REPORT: {base_name}")
    print("="*85)
    header = f"{'Status':<15} | {'T1 ID (Real)':<15} | {'T2 ID (Real)':<15} | {'Growth %':<10} | {'Verification'}"
    print(header)
    print("-" * 85)

    correct_matches = 0
    total_persistent = 0

    for m in matches:
        status = m['status']

        if status == "Persistent":
            total_persistent += 1
            # Get Ground Truth IDs to verify AI matching
            id1 = get_gt_id(masks_t1[m['t1_idx']], T1_META)
            id2 = get_gt_id(masks_t2[m['t2_idx']], T2_META)

            # Check if AI matched the SAME lesion name
            is_correct = (id1 == id2 and id1 != "Unknown")
            if is_correct: correct_matches += 1

            val_text = "CORRECT" if is_correct else "MISMATCH"
            growth_str = f"{m['growth_pct']:>+.1f}%"

            print(f"{status:<15} | {id1:<15} | {id2:<15} | {growth_str:<10} | {val_text}")

        elif status == "New Lesion":
            id2 = get_gt_id(masks_t2[m['t2_idx']], T2_META)
            print(f"{status:<15} | {'[NONE]':<15} | {id2:<15} | {'N/A':<10} | New Arrival")

        elif status == "Disappeared":
            id1 = get_gt_id(masks_t1[m['t1_idx']], T1_META)
            print(f"{status:<15} | {id1:<15} | {'[GONE]':<15} | -100.0%   | Resolved")

    print("-" * 85)
    if total_persistent > 0:
        accuracy = (correct_matches / total_persistent) * 100
        print(f"SUMMARY: Matched {correct_matches}/{total_persistent} existing lesions accurately ({accuracy:.1f}% Match Accuracy)")
    print("="*85 + "\n")

# TEST IT
run_full_analysis("001_WFemale")